# Apply Author Work Curations

Applies `add` and `remove` work-curations from `openalex.authors.author_works_curations` onto `openalex.works.work_authors`. Runs after `MatchAuthors` (which finalizes `author_id`) and before `UpdateWorkAuthorships` (which materializes the final authorships struct).

## Semantics (per oxjob #168 PLAN, resolved 2026-05-07)

- **`add(B, W)` = transfer**: rewrite `work_authors.author_id` to B at the position of W whose parsed `raw_author_name` best matches B's parsed name. Detaches whatever profile previously held that position. NOT permanent on its own.
- **`remove(A, W)` = sticky disclaim**: NULL `work_authors.author_id` at any position of W where it currently equals A. Next `MatchAuthors` cycle re-picks; the block-list filter in `MatchAuthors.blocked_candidates` prevents A from being re-selected.

## Position matching for `add`

The curation row carries no `author_sequence`. We pick the position of W whose parsed name scores highest against B's parsed name. Score floor is last-name match; pairs with no last-name overlap are silently skipped (visible only via `add_curations_total` minus `add_positions_resolved` in the verify cell).

## Affected works queue

Every work_id touched (added or removed) is inserted into `openalex.institutions.curated_work_ids_pending_sync` so `UpdateWorkAuthorships` re-flows them on its next run.

## Stage curated add positions

For each (curated author B, work W) in `curated_add_work_ids`, pick the best-matching position k of W. Cleared and rebuilt every run.

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.authors.author_curation_add_positions AS
WITH add_curations AS (
    SELECT
        author_id AS curated_author_id,
        EXPLODE(curated_add_work_ids) AS work_id
    FROM openalex.authors.author_works_curations
    WHERE curated_add_work_ids IS NOT NULL
      AND SIZE(curated_add_work_ids) > 0
),
curator_names AS (
    SELECT
        ac.curated_author_id,
        ac.work_id,
        afm.first         AS curator_first,
        afm.first_initial AS curator_first_initial,
        afm.middle        AS curator_middle,
        afm.last          AS curator_last
    FROM add_curations ac
    LEFT JOIN openalex.authors.authors_for_matching afm
        ON ac.curated_author_id = afm.author_id
),
work_position_names AS (
    SELECT
        wa.work_id,
        wa.author_sequence,
        wa.author_id AS current_author_id,
        pn.parsed_name.first                       AS pos_first,
        SUBSTRING(pn.parsed_name.first, 1, 1)      AS pos_first_initial,
        pn.parsed_name.middle                      AS pos_middle,
        pn.parsed_name.last                        AS pos_last
    FROM openalex.works.work_authors wa
    LEFT JOIN openalex.authors.author_names pn
        ON TRIM(wa.raw_author_name) = pn.raw_author_name
),
scored AS (
    SELECT
        cn.curated_author_id,
        cn.work_id,
        wpn.author_sequence,
        wpn.current_author_id,
        CASE
            WHEN cn.curator_last IS NULL OR wpn.pos_last IS NULL THEN 0
            WHEN cn.curator_last <> wpn.pos_last THEN 0
            -- last name matches; rank by first/middle granularity
            WHEN cn.curator_first = wpn.pos_first
             AND cn.curator_middle = wpn.pos_middle
             AND LENGTH(cn.curator_first) > 1                                       THEN 100
            WHEN cn.curator_first = wpn.pos_first
             AND LENGTH(cn.curator_first) > 1                                       THEN 80
            WHEN cn.curator_first_initial = wpn.pos_first_initial
             AND SUBSTRING(cn.curator_middle, 1, 1) = SUBSTRING(wpn.pos_middle, 1, 1)
             AND cn.curator_first_initial IS NOT NULL                               THEN 60
            WHEN cn.curator_first_initial = wpn.pos_first_initial
             AND cn.curator_first_initial IS NOT NULL                               THEN 40
            ELSE 20  -- last name only
        END AS match_score
    FROM curator_names cn
    JOIN work_position_names wpn
        ON cn.work_id = wpn.work_id
)
SELECT curated_author_id, work_id, author_sequence, current_author_id, match_score
FROM (
    SELECT
        curated_author_id, work_id, author_sequence, current_author_id, match_score,
        ROW_NUMBER() OVER (
            PARTITION BY curated_author_id, work_id
            ORDER BY match_score DESC, author_sequence ASC
        ) AS rn
    FROM scored
    WHERE match_score >= 20
)
WHERE rn = 1

## Apply add curations to work_authors

MERGE the staged positions onto `work_authors`, rewriting `author_id` at the chosen position.

In [ ]:
%sql
MERGE INTO openalex.works.work_authors AS target
USING openalex.authors.author_curation_add_positions AS source
ON target.work_id = source.work_id
   AND target.author_sequence = source.author_sequence
WHEN MATCHED AND (target.author_id IS NULL OR target.author_id <> source.curated_author_id) THEN
  UPDATE SET
    target.author_id = source.curated_author_id,
    target.updated_at = current_timestamp()

## Apply remove curations to work_authors

NULL the `author_id` at any (work_id, author_sequence) where it currently equals the curated remove pair. Next `MatchAuthors` cycle re-picks (with the (work_id, author_id) pair blocked from candidacy via the block-list filter in `MatchAuthors.blocked_candidates`).

In [ ]:
%sql
MERGE INTO openalex.works.work_authors AS target
USING (
    SELECT
        author_id AS curated_author_id,
        EXPLODE(curated_remove_work_ids) AS work_id
    FROM openalex.authors.author_works_curations
    WHERE curated_remove_work_ids IS NOT NULL
      AND SIZE(curated_remove_work_ids) > 0
) AS source
ON target.work_id = source.work_id
   AND target.author_id = source.curated_author_id
WHEN MATCHED THEN
  UPDATE SET
    target.author_id = NULL,
    target.updated_at = current_timestamp()

## Queue affected work_ids for re-sync

Insert every work_id touched by add or remove curations into `curated_work_ids_pending_sync`. `UpdateWorkAuthorships` picks them up (via its `WHERE id IN (SELECT work_id FROM curated_work_ids_pending_sync)` clause) and truncates the queue at the end of its run.

In [ ]:
%sql
INSERT INTO openalex.institutions.curated_work_ids_pending_sync
SELECT DISTINCT
    work_id,
    NULL AS curated_ras,
    current_timestamp() AS added_datetime
FROM (
    SELECT EXPLODE(curated_add_work_ids) AS work_id
    FROM openalex.authors.author_works_curations
    WHERE curated_add_work_ids IS NOT NULL
      AND SIZE(curated_add_work_ids) > 0
    UNION
    SELECT EXPLODE(curated_remove_work_ids) AS work_id
    FROM openalex.authors.author_works_curations
    WHERE curated_remove_work_ids IS NOT NULL
      AND SIZE(curated_remove_work_ids) > 0
)

## Verify

In [ ]:
%sql
SELECT
    (SELECT COUNT(*) FROM (
        SELECT EXPLODE(curated_add_work_ids) AS w
        FROM openalex.authors.author_works_curations
        WHERE curated_add_work_ids IS NOT NULL
     ))                                                                          AS add_curations_total,
    (SELECT COUNT(*) FROM openalex.authors.author_curation_add_positions)        AS add_positions_resolved,
    (SELECT COUNT(*) FROM (
        SELECT EXPLODE(curated_remove_work_ids) AS w
        FROM openalex.authors.author_works_curations
        WHERE curated_remove_work_ids IS NOT NULL
     ))                                                                          AS remove_curations_total

In [ ]:
%sql
SELECT wa.work_id, wa.author_sequence, wa.author_id, wa.raw_author_name, wa.updated_at
FROM openalex.works.work_authors wa
WHERE wa.work_id IN (
    SELECT EXPLODE(curated_add_work_ids)
    FROM openalex.authors.author_works_curations
    WHERE curated_add_work_ids IS NOT NULL
    UNION
    SELECT EXPLODE(curated_remove_work_ids)
    FROM openalex.authors.author_works_curations
    WHERE curated_remove_work_ids IS NOT NULL
)
ORDER BY wa.updated_at DESC
LIMIT 100